# Notebook 03 — Full SCM Identification

**Across the Water: French Nuclear Unplanned Outages and GB Power Prices**

**Conditional on Notebook 02 DA gate passing.**

Full Structural Causal Model estimation of the unplanned outage → GB DA price
causal effect. Includes all pre-registered adversarial checks and falsification
conditions.

---

### Causal structure

```
Unplanned outage increment (Z)
         ↓
French export capacity (X) ←── confounders W:
         ↓                       TTF gas spot
  IFA/IFA2 flow (M)              FR temp deviation
         ↓                       DE wind generation
  GB DA/ID price (Y) ←──────────┘
```

All confounders in W are **directly observed** and enter as regression controls.
Estimator: OLS panel with Newey-West HAC SE (12 lags).


> 📋 **NOT RUN** — Pre-registered but not executed.
> The DA hard gate (Notebook 02) failed before this gate was reached.
> Published unmodified to demonstrate pre-registration discipline.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from src import log, load, DATA_DIR, MAIN_START, MAIN_END
from src.utils import add_season_col

pd.set_option("display.float_format", "{:.4f}".format)
print("Setup complete.")


---
## 1. Load Panel (reusing 02 build logic)

In [ ]:
# Reload all datasets and rebuild panel
# (duplicated from 02 for notebook independence)

gb_da = load("elexon_da_prices")[["gb_da_price"]]
gb_da.index = pd.to_datetime(gb_da.index, utc=True)

entso = load("entso_unavailability")
entso.index = pd.to_datetime(entso.index, utc=True)

def daily_outage_mw(df, outage_type):
    subset = df[df["outage_type"] == outage_type]
    return (
        subset.groupby(subset.index.normalize())["unavailable_mw"]
        .sum().rename(f"{outage_type}_outage_mw")
    )

neso = load("neso_demand")
neso.index = pd.to_datetime(neso.index, utc=True)
neso_d = neso.resample("D").mean()
neso_d["wind_forecast_error"]   = neso_d.get("wind_actual_mw", np.nan) - neso_d.get("wind_forecast_mw", np.nan)
neso_d["demand_forecast_error"] = neso_d.get("demand_actual_mw", np.nan) - neso_d.get("demand_forecast_mw", np.nan)
ifa_cols = [c for c in ["ifa1_flow_mw", "ifa2_flow_mw"] if c in neso_d.columns]
neso_d["ifa_flow_mw"] = neso_d[ifa_cols].sum(axis=1) if ifa_cols else np.nan

fr_temp = load("fr_temperature")[["fr_temp_deviation"]]
fr_temp.index = pd.to_datetime(fr_temp.index, utc=True)

try:
    ttf = load("ttf_spot")[["ttf_spot_eur_mwh"]]
    ttf.index = pd.to_datetime(ttf.index, utc=True)
except FileNotFoundError:
    ttf = pd.DataFrame(columns=["ttf_spot_eur_mwh"])

panel = gb_da.copy()
panel = panel.join(daily_outage_mw(entso, "unplanned"), how="left")
panel = panel.join(daily_outage_mw(entso, "planned"),   how="left")
panel = panel.join(neso_d[["wind_forecast_error", "demand_forecast_error", "ifa_flow_mw"]], how="left")
panel = panel.join(fr_temp, how="left")
if not ttf.empty:
    panel = panel.join(ttf, how="left")
else:
    panel["ttf_spot_eur_mwh"] = np.nan

panel["unplanned_outage_mw"] = panel["unplanned_outage_mw"].fillna(0)
panel["planned_outage_mw"]   = panel["planned_outage_mw"].fillna(0)
panel = panel[panel.index >= pd.Timestamp(MAIN_START, tz="UTC")]
panel = add_season_col(panel)
panel["year"] = panel.index.year

print(f"Panel: {len(panel)} daily observations")


---
## 2. Full SCM Estimation — Model A and Model B

Separate IFA1 and IFA2 channel regressions as specified in README.


In [ ]:
def scm_regression(panel, extra_controls=None, label="Model A"):
    """
    Full SCM OLS regression.
    extra_controls: list of additional column names (e.g. ['fr_da_price'])
    """
    controls = [
        "unplanned_outage_mw",
        "planned_outage_mw",
        "wind_forecast_error",
        "demand_forecast_error",
        "ifa_flow_mw",
        "fr_temp_deviation",
        "ttf_spot_eur_mwh",
    ]
    if extra_controls:
        controls += [c for c in extra_controls if c in panel.columns]
    
    available = [c for c in controls if c in panel.columns]
    season_dummies = pd.get_dummies(panel["season"], prefix="s", drop_first=True)
    year_dummies   = pd.get_dummies(panel["year"],   prefix="y", drop_first=True)
    
    X = pd.concat([panel[available], season_dummies, year_dummies], axis=1).astype(float)
    X = sm.add_constant(X)
    y = panel["gb_da_price"].astype(float)
    
    mask = X.notna().all(axis=1) & y.notna()
    model = sm.OLS(y[mask], X[mask]).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    return model, label

model_a, label_a = scm_regression(panel, label="Model A (total effect)")
print(f"Model A: β = {model_a.params['unplanned_outage_mw']*1000:.4f} £/MWh per GW")
print(f"         p = {model_a.pvalues['unplanned_outage_mw']:.4f}")
print(f"         R² = {model_a.rsquared:.4f}, N = {model_a.nobs:.0f}")
print()
print(model_a.summary(title="Model A — Full SCM (HAC SE, 12 lags)"))


---
## 3. Pre-registered Adversarial Checks

In [ ]:
# ── 3a. Seasonal stability ────────────────────────────────────────────────────
print("3a. Seasonal stability — β should be larger in Winter")
print("    (higher GB demand, tighter merit order)")
print()

results_seasonal = {}
for season_name in ["Winter", "Spring", "Summer", "Autumn"]:
    s_panel = panel[panel["season"] == season_name]
    if len(s_panel) < 50:
        print(f"  {season_name}: insufficient data ({len(s_panel)} obs) — skipped")
        continue
    model_s, _ = scm_regression(s_panel, label=season_name)
    b = model_s.params.get("unplanned_outage_mw", float("nan")) * 1000
    p = model_s.pvalues.get("unplanned_outage_mw", float("nan"))
    results_seasonal[season_name] = {"beta_gw": b, "p": p, "n": model_s.nobs}
    print(f"  {season_name:<8}: β = {b:.3f} £/MWh per GW  (p={p:.3f}, n={model_s.nobs:.0f})")

print()
if results_seasonal:
    winter_b = results_seasonal.get("Winter", {}).get("beta_gw", float("nan"))
    summer_b = results_seasonal.get("Summer", {}).get("beta_gw", float("nan"))
    if winter_b > summer_b:
        print("✅  Winter β > Summer β — consistent with demand-driven mechanism")
    else:
        print("⚠️  Winter β ≤ Summer β — unexpected; review mechanism interpretation")


In [ ]:
# ── 3b. Size monotonicity ─────────────────────────────────────────────────────
print("3b. Size monotonicity — larger outages should → larger price effects")
print()

panel["outage_bin"] = pd.cut(
    panel["unplanned_outage_mw"],
    bins=[0, 1, 500, 1000, 2000, panel["unplanned_outage_mw"].max()+1],
    labels=["0 (control)", "1–500 MW", "500–1000 MW", "1000–2000 MW", ">2000 MW"],
    right=True,
)
bin_means = panel.groupby("outage_bin")["gb_da_price"].mean()
print(bin_means.to_string())
print()
print("Conditional on: this is raw, not SCM-adjusted — use as directional check only.")


In [ ]:
# ── 3c. Planned vs unplanned comparison (falsification) ───────────────────────
print("3c. Planned outage effect vs unplanned — instrument validity check")
print("    Pre-registered: planned β should be near zero (market pre-prices these).")
print()

controls_no_planned = [
    "unplanned_outage_mw",
    "wind_forecast_error", "demand_forecast_error",
    "ifa_flow_mw", "fr_temp_deviation", "ttf_spot_eur_mwh",
]
available = [c for c in controls_no_planned if c in panel.columns]
season_d  = pd.get_dummies(panel["season"], prefix="s", drop_first=True)
year_d    = pd.get_dummies(panel["year"],   prefix="y", drop_first=True)

# Add planned as treatment instead
X_planned = pd.concat([
    panel[["planned_outage_mw"] + available[1:]],
    season_d, year_d
], axis=1).astype(float)
X_planned = sm.add_constant(X_planned)
y = panel["gb_da_price"].astype(float)
mask = X_planned.notna().all(axis=1) & y.notna()

model_planned = sm.OLS(y[mask], X_planned[mask]).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
b_planned = model_planned.params.get("planned_outage_mw", float("nan")) * 1000
p_planned = model_planned.pvalues.get("planned_outage_mw", float("nan"))

b_unplanned = model_a.params.get("unplanned_outage_mw", float("nan")) * 1000
p_unplanned = model_a.pvalues.get("unplanned_outage_mw", float("nan"))

print(f"  Unplanned outage β: {b_unplanned:.4f} £/MWh per GW  (p={p_unplanned:.4f})")
print(f"  Planned outage β:   {b_planned:.4f} £/MWh per GW  (p={p_planned:.4f})")
print()

if abs(b_planned) < abs(b_unplanned) * 0.5 or p_planned > 0.15:
    print("✅  Planned β substantially smaller — consistent with market pre-pricing planned outages.")
    print("   Instrument validity supported.")
else:
    print("⚠️  Planned β comparable to unplanned — pre-registration falsification 3 triggered.")
    print("   Instrument may be partially invalid. Interpret results with caution.")


In [ ]:
# ── 3d. IFA flow mediation check ──────────────────────────────────────────────
print("3d. IFA flow mediation check")
print("    Pre-registered: outage should reduce IFA flows.")
print("    Note: flows depend on price spreads + operational decisions —")
print("    non-mediation is a plausible outcome, not a surprise.")
print()

X_flow = pd.concat([
    panel[["unplanned_outage_mw", "planned_outage_mw", "fr_temp_deviation", "ttf_spot_eur_mwh"]],
    season_d, year_d
], axis=1).astype(float)
X_flow = sm.add_constant(X_flow)
y_flow = panel["ifa_flow_mw"].astype(float)
mask_f = X_flow.notna().all(axis=1) & y_flow.notna()

if mask_f.sum() > 50:
    model_flow = sm.OLS(y_flow[mask_f], X_flow[mask_f]).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    b_flow = model_flow.params.get("unplanned_outage_mw", float("nan"))
    p_flow = model_flow.pvalues.get("unplanned_outage_mw", float("nan"))
    
    print(f"  IFA flow ~ unplanned_outage_mw: β = {b_flow:.4f} MW flow per MW outage  (p={p_flow:.4f})")
    if b_flow < 0:
        print("  → Negative coefficient: outages reduce IFA flows (expected direction).")
    else:
        print("  → Positive or zero coefficient: IFA flows not mediated as expected.")
        print("    This may indicate France is operating at low net export before the trip.")
else:
    print("  Insufficient data for IFA flow regression.")


---
## 4. Summary Coefficient Plot

In [ ]:
# Collect seasonal + overall coefficients
plot_labels = ["Overall"]
plot_betas  = [model_a.params.get("unplanned_outage_mw", float("nan")) * 1000]
plot_lo     = [(model_a.params.get("unplanned_outage_mw",0) - 1.96*model_a.bse.get("unplanned_outage_mw",0)) * 1000]
plot_hi     = [(model_a.params.get("unplanned_outage_mw",0) + 1.96*model_a.bse.get("unplanned_outage_mw",0)) * 1000]

for sname in ["Winter", "Spring", "Summer", "Autumn"]:
    s_panel = panel[panel["season"] == sname]
    if len(s_panel) < 50:
        continue
    model_s, _ = scm_regression(s_panel, label=sname)
    b  = model_s.params.get("unplanned_outage_mw", float("nan")) * 1000
    se = model_s.bse.get("unplanned_outage_mw", 0) * 1000
    plot_labels.append(sname)
    plot_betas.append(b)
    plot_lo.append(b - 1.96*se)
    plot_hi.append(b + 1.96*se)

fig, ax = plt.subplots(figsize=(9, 4))
y_pos = range(len(plot_labels))
colors = ["tomato"] + ["steelblue"] * (len(plot_labels)-1)
for i, (label, beta, lo, hi, color) in enumerate(zip(plot_labels, plot_betas, plot_lo, plot_hi, colors)):
    ax.barh(i, beta, color=color, alpha=0.75)
    ax.errorbar(beta, i, xerr=[[beta-lo],[hi-beta]], fmt="none", color="black", capsize=4, lw=1.5)
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_labels)
ax.set_xlabel("β: £/MWh per GW of unplanned outage (95% CI)")
ax.set_title("SCM Causal Effect — Overall and by Season (Model A)")
plt.tight_layout()
plt.savefig(DATA_DIR / "plot_03_scm_coef.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Summary

Full SCM estimation complete. Key outputs:
- Model A: total causal effect (IFA1 + IFA2 channels)
- Model B: residual after FR DA clear (if FR DA data available)
- Seasonal stability check
- Size monotonicity check
- Planned vs unplanned falsification
- IFA flow mediation check

Proceed to **Notebook 04** (quantile model) if all adversarial checks pass.
